I turned the raw FDA tables into one clean table of 477 reports (one row per report) plus problem, harm and narrative tables. I removed duplicates, defined serious and severe outcomes from patient outcome codes, translated FDA codes into plain categories, and standardised 142 manufacturer spellings into 108 companies.



# Case Study 1 – **Cleaning**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
BASE = '/content/drive/MyDrive/cs1-device-safety'
F = f'{BASE}/data/filtered'

def load(name):
    df = pd.read_parquet(f'{F}/{name}.parquet')
    return df.apply(lambda s: s.str.strip() if s.dtype == object else s)

devices, master, patients = load('devices'), load('master'), load('patients')
dev_problems, pat_problems, narratives = load('dev_problems'), load('pat_problems'), load('narratives')
dev_lookup, pat_lookup = load('dev_lookup'), load('pat_lookup')

DEVICE_GROUPS = {
    'GZJ': 'TENS (prescription)',
    'NUH': 'TENS (over-the-counter)',
    'IPF': 'Powered muscle stimulator',
    'PHL': 'Powered exoskeleton',
    'BXB': 'Powered exercise equipment (incl. CPM)',
}

for name, df in [('devices', devices), ('master', master), ('patients', patients),
                 ('dev_problems', dev_problems), ('pat_problems', pat_problems),
                 ('narratives', narratives), ('dev_lookup', dev_lookup), ('pat_lookup', pat_lookup)]:
    print(f'{name:13s} {df.shape[0]:6,} rows  {df.shape[1]:3} columns')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
devices          480 rows   34 columns
master           477 rows   86 columns
patients         476 rows   10 columns
dev_problems     851 rows    2 columns
pat_problems   1,388 rows    5 columns
narratives       793 rows    6 columns
dev_lookup       885 rows    4 columns
pat_lookup     1,172 rows    4 columns


Step 9.1 – Remove duplicate rows. Each table has its own rule for what counts as a duplicate; when a report appears in both the historical and current-year files, keep the newer copy.

In [3]:
before = {'devices': len(devices), 'master': len(master), 'patients': len(patients),
          'dev_problems': len(dev_problems), 'pat_problems': len(pat_problems)}

devices = devices.drop_duplicates(['MDR_REPORT_KEY', 'DEVICE_SEQUENCE_NO'], keep='last')
master = master.drop_duplicates('MDR_REPORT_KEY', keep='last')
patients = patients.drop_duplicates(['MDR_REPORT_KEY', 'PATIENT_SEQUENCE_NUMBER'], keep='last')
dev_problems = dev_problems.drop_duplicates(['MDR_REPORT_KEY', 'DEVICE_PROBLEM_CODE'])
pat_problems = pat_problems.drop_duplicates(['MDR_REPORT_KEY', 'PATIENT_SEQUENCE_NO', 'PROBLEM_CODE'])

after = {'devices': len(devices), 'master': len(master), 'patients': len(patients),
         'dev_problems': len(dev_problems), 'pat_problems': len(pat_problems)}
print(pd.DataFrame({'before': before, 'after': after}))

              before  after
devices          480    480
master           477    477
patients         476    476
dev_problems     851    548
pat_problems    1388    721


Step 9.2 – Inspect the fields used to define serious harm before writing the rule.

In [4]:
print('MASTER columns:\n', list(master.columns), '\n')
print('DEVICE columns:\n', list(devices.columns), '\n')
print('PATIENT columns:\n', list(patients.columns), '\n')

event_col = [c for c in master.columns if c == 'EVENT_TYPE' or c.endswith('EVENT_TYPE')]
print('Event type column found:', event_col)
if event_col:
    print(master[event_col[0]].value_counts(dropna=False), '\n')

print('Patient outcome values (top 20):')
print(patients['SEQUENCE_NUMBER_OUTCOME'].value_counts(dropna=False).head(20), '\n')

print('Report source:')
print(master['REPORT_SOURCE_CODE'].value_counts(dropna=False))

MASTER columns:
 ['MDR_REPORT_KEY', 'EVENT_KEY', 'REPORT_NUMBER', 'REPORT_SOURCE_CODE', 'MANUFACTURER_LINK_FLAG_', 'NUMBER_DEVICES_IN_EVENT', 'NUMBER_PATIENTS_IN_EVENT', 'DATE_RECEIVED', 'ADVERSE_EVENT_FLAG', 'PRODUCT_PROBLEM_FLAG', 'DATE_REPORT', 'DATE_OF_EVENT', 'REPROCESSED_AND_REUSED_FLAG', 'REPORTER_OCCUPATION_CODE', 'HEALTH_PROFESSIONAL', 'INITIAL_REPORT_TO_FDA', 'DATE_FACILITY_AWARE', 'REPORT_DATE', 'REPORT_TO_FDA', 'DATE_REPORT_TO_FDA', 'EVENT_LOCATION', 'DATE_REPORT_TO_MANUFACTURER', 'MANUFACTURER_CONTACT_T_NAME', 'MANUFACTURER_CONTACT_F_NAME', 'MANUFACTURER_CONTACT_L_NAME', 'MANUFACTURER_CONTACT_STREET_1', 'MANUFACTURER_CONTACT_STREET_2', 'MANUFACTURER_CONTACT_CITY', 'MANUFACTURER_CONTACT_STATE', 'MANUFACTURER_CONTACT_ZIP_CODE', 'MANUFACTURER_CONTACT_ZIP_EXT', 'MANUFACTURER_CONTACT_COUNTRY', 'MANUFACTURER_CONTACT_POSTAL', 'MANUFACTURER_CONTACT_AREA_CODE', 'MANUFACTURER_CONTACT_EXCHANGE', 'MANUFACTURER_CONTACT_PHONE_NO', 'MANUFACTURER_CONTACT_EXTENSION', 'MANUFACTURER_CONTACT_

Step 9.3 – Define serious outcome (D, L, H, S, R, C) and severe outcome (D, L, H, S) per report, from patient outcome codes.

In [5]:
OUTCOMES = {'D': 'death', 'L': 'life_threatening', 'H': 'hospitalised',
            'S': 'disability', 'R': 'required_intervention',
            'C': 'congenital_anomaly', 'O': 'other'}

def codes(value):
    if value is None or pd.isna(value):
        return set()
    return {x.strip() for x in str(value).replace(',', ';').split(';') if x.strip()}

# 1. Turn each patient's outcome text into a set of letters, then combine patients per report
patients['codes'] = patients['SEQUENCE_NUMBER_OUTCOME'].apply(codes)
per_report = patients.groupby('MDR_REPORT_KEY')['codes'].agg(lambda x: set().union(*x))

# 2. One row per report
harm = pd.DataFrame(index=master['MDR_REPORT_KEY'])
harm['outcome_codes'] = per_report.reindex(harm.index).apply(lambda s: s if isinstance(s, set) else set())
for code, name in OUTCOMES.items():
    harm[f'out_{name}'] = harm['outcome_codes'].apply(lambda s: code in s)
harm['outcome_reported'] = harm['outcome_codes'].apply(lambda s: len(s & set(OUTCOMES)) > 0)
harm['serious_outcome'] = harm['outcome_codes'].apply(lambda s: len(s & set('DLHSRC')) > 0)
harm['severe_outcome'] = harm['outcome_codes'].apply(lambda s: len(s & set('DLHS')) > 0)

# 3. Add device group and event type
dev_first = devices.drop_duplicates('MDR_REPORT_KEY').set_index('MDR_REPORT_KEY')
harm['device_group'] = dev_first['DEVICE_REPORT_PRODUCT_CODE'].map(DEVICE_GROUPS).reindex(harm.index)
harm['event_type'] = (master.set_index('MDR_REPORT_KEY')['EVENT_TYPE'].reindex(harm.index)
                      .map({'IN': 'Injury', 'M': 'Malfunction', 'D': 'Death', 'O': 'Other'})
                      .fillna('Not reported'))

# 4. Summary per device group
summary = harm.groupby('device_group').agg(
    reports=('event_type', 'size'),
    outcome_reported=('outcome_reported', 'sum'),
    serious=('serious_outcome', 'sum'),
    severe=('severe_outcome', 'sum'))
summary['serious_%_of_reports'] = (100 * summary['serious'] / summary['reports']).round(1)
print(summary, '\n')

# 5. How many reports are malfunction summaries?
print(master['SUMMARY_REPORT'].value_counts(dropna=False))

                                        reports  outcome_reported  serious  \
device_group                                                                 
Powered exercise equipment (incl. CPM)       40                21       15   
Powered exoskeleton                          54                51       42   
Powered muscle stimulator                    73                46       19   
TENS (over-the-counter)                      97                75       30   
TENS (prescription)                         213               164       54   

                                        severe  serious_%_of_reports  
device_group                                                          
Powered exercise equipment (incl. CPM)      11                  37.5  
Powered exoskeleton                          7                  77.8  
Powered muscle stimulator                   11                  26.0  
TENS (over-the-counter)                     18                  30.9  
TENS (prescription)        

In [6]:
fields = [
    (master, 'EVENT_LOCATION'),
    (master, 'REPORTER_OCCUPATION_CODE'),
    (master, 'REMEDIAL_ACTION'),
    (devices, 'DEVICE_OPERATOR'),
    (devices, 'DEVICE_EVALUATED_BY_MANUFACTUR'),
    (devices, 'DEVICE_AVAILABILITY'),
    (devices, 'DEVICE_AGE_TEXT'),
    (patients, 'PATIENT_AGE'),
    (patients, 'PATIENT_SEX'),
]
for df, col in fields:
    print(f'===== {col}  ({df[col].nunique()} different values)')
    print(df[col].value_counts(dropna=False).head(15), '\n')

===== EVENT_LOCATION  (7 different values)
EVENT_LOCATION
None    161
I       120
004      50
NI       49
002      48
000      43
001       5
005       1
Name: count, dtype: int64 

===== REPORTER_OCCUPATION_CODE  (22 different values)
REPORTER_OCCUPATION_CODE
000    217
305     99
100     51
111     19
003     18
001     12
002     12
501     11
999      9
500      6
303      5
306      4
0HP      3
505      2
401      2
Name: count, dtype: int64 

===== REMEDIAL_ACTION  (17 different values)
REMEDIAL_ACTION
None     419
OT         9
PM         8
RL         7
IN         6
RC,NO      5
NO,RC      4
RP         4
NO         3
NO,IN      3
MA         2
IN,NO      1
IN,RL      1
RL,OT      1
IN,OT      1
Name: count, dtype: int64 

===== DEVICE_OPERATOR  (9 different values)
DEVICE_OPERATOR
0LP     172
0HP     116
None     77
I        44
0        29
111      22
100      11
303       5
000       3
002       1
Name: count, dtype: int64 

===== DEVICE_EVALUATED_BY_MANUFACTUR  (4 different val

In [7]:
import re, numpy as np

def location(c):
    if c in ('002', '810'): return 'Home'
    if c == '001' or (c and c.isdigit() and 500 <= int(c) <= 511): return 'Hospital'
    if c in ('004', '005', '006', '704') or (c and c.isdigit() and 600 <= int(c) <= 617):
        return 'Clinic / outpatient / rehab'
    if c == '003' or (c and c.isdigit() and 700 <= int(c) <= 705): return 'Nursing / long-term care'
    if c == '000' or (c and c.isdigit() and 820 <= int(c) <= 836): return 'Other'
    return 'Not reported'

LAY = {'0LP', '305', '306', '307', '300', '003'}
HEALTH_PRO = {'0HP', '1', '2', '001', '002', '301', '302', '303', '304'} | {str(i) for i in range(100, 118)}

def operator(c):
    if c in LAY: return 'Patient / lay user'
    if c in HEALTH_PRO: return 'Health professional'
    if c in ('0', '000'): return 'Other'
    return 'Not reported'

def reporter(c):
    if c in LAY: return 'Patient / family'
    if c in HEALTH_PRO: return 'Clinician'
    if c and c.isdigit() and (400 <= int(c) <= 405 or 500 <= int(c) <= 505): return 'Facility staff / admin'
    return 'Other / not reported'

def to_years(v):
    if v is None or pd.isna(v): return np.nan
    m = re.match(r'^\s*(\d+(?:\.\d+)?)\s*(YR|MO|WK|DA|HR)', str(v).upper())
    if not m: return np.nan
    return round(float(m.group(1)) / {'YR': 1, 'MO': 12, 'WK': 52, 'DA': 365, 'HR': 8760}[m.group(2)], 1)

print(location('810'), '|', operator('0LP'), '|', reporter('111'), '|', to_years('21 MO'), '|', to_years('DA'))

Home | Patient / lay user | Clinician | 1.8 | nan


In [8]:
m = master.set_index('MDR_REPORT_KEY')
p_first = patients.drop_duplicates('MDR_REPORT_KEY').set_index('MDR_REPORT_KEY')
idx = harm.index
get = lambda df, col: df[col].reindex(idx)

reports = harm.drop(columns=['outcome_codes']).copy()
reports['product_code'] = get(dev_first, 'DEVICE_REPORT_PRODUCT_CODE')
reports['date_received'] = pd.to_datetime(get(m, 'DATE_RECEIVED'), format='%m/%d/%Y', errors='coerce')
reports['year'] = reports['date_received'].dt.year
reports['event_key'] = get(m, 'EVENT_KEY')
reports['report_source'] = get(m, 'REPORT_SOURCE_CODE').map(
    {'M': 'Manufacturer', 'U': 'Hospital / user facility',
     'P': 'Voluntary (patient or clinician)', 'D': 'Distributor'}).fillna('Not reported')
reports['event_location'] = get(m, 'EVENT_LOCATION').apply(location)
reports['operator'] = get(dev_first, 'DEVICE_OPERATOR').apply(operator)
reports['reporter'] = get(m, 'REPORTER_OCCUPATION_CODE').apply(reporter)
reports['home_or_lay_use'] = (reports['event_location'] == 'Home') | (reports['operator'] == 'Patient / lay user')

actions = get(m, 'REMEDIAL_ACTION').apply(codes)
reports['corrective_action'] = actions.apply(lambda s: len(s - {'*'}) > 0)
reports['recall'] = actions.apply(lambda s: 'RC' in s)
reports['device_returned'] = get(dev_first, 'DEVICE_AVAILABILITY').eq('R')

reports['manufacturer_raw'] = get(dev_first, 'MANUFACTURER_D_NAME')
reports['brand'] = get(dev_first, 'BRAND_NAME')
reports['device_age_years'] = get(dev_first, 'DEVICE_AGE_TEXT').apply(to_years)
reports['patient_age'] = get(p_first, 'PATIENT_AGE').apply(to_years)

# ---- Checks ----
print('Shape:', reports.shape, '| unparsed dates:', reports['date_received'].isna().sum(), '\n')
for col in ['event_location', 'operator', 'reporter', 'report_source']:
    print(reports[col].value_counts(), '\n')
print('Raw location codes that became "Not reported":')
print(get(m, 'EVENT_LOCATION')[reports['event_location'] == 'Not reported'].value_counts(dropna=False), '\n')
print('Raw operator codes that became "Not reported":')
print(get(dev_first, 'DEVICE_OPERATOR')[reports['operator'] == 'Not reported'].value_counts(dropna=False), '\n')
print('Home or lay use:', reports['home_or_lay_use'].sum(),
      '| corrective action:', reports['corrective_action'].sum(),
      '| recalls:', reports['recall'].sum(),
      '| patient age known:', reports['patient_age'].notna().sum())

Shape: (477, 28) | unparsed dates: 0 

event_location
Not reported                   330
Clinic / outpatient / rehab     51
Home                            48
Other                           43
Hospital                         5
Name: count, dtype: int64 

operator
Patient / lay user     171
Health professional    155
Not reported           119
Other                   32
Name: count, dtype: int64 

reporter
Other / not reported      227
Patient / family          121
Clinician                 107
Facility staff / admin     22
Name: count, dtype: int64 

report_source
Manufacturer                        218
Voluntary (patient or clinician)    144
Distributor                         101
Hospital / user facility             14
Name: count, dtype: int64 

Raw location codes that became "Not reported":
EVENT_LOCATION
None    161
I       120
NI       49
Name: count, dtype: int64 

Raw operator codes that became "Not reported":
DEVICE_OPERATOR
None    77
I       42
Name: count, dtype: int64 



Step 9.6 – Standardise manufacturer names: automatic cleaning, then manual review.

In [9]:
SUFFIX = r'\b(INC|INCORPORATED|LLC|L L C|LTD|LIMITED|CO|CORP|CORPORATION|COMPANY|GMBH|AG|S A|SA|PLC|LP)\b'

def clean_name(n):
    if n is None or pd.isna(n):
        return 'Not reported'
    s = re.sub(r'[^\w\s&]', ' ', str(n).upper())
    s = re.sub(SUFFIX, ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s or 'Not reported'

reports['manufacturer'] = reports['manufacturer_raw'].apply(clean_name)
print('Distinct names:', reports['manufacturer_raw'].nunique(), 'raw ->',
      reports['manufacturer'].nunique(), 'after cleaning\n')

review = (reports.groupby(['device_group', 'manufacturer'])
          .agg(reports=('manufacturer_raw', 'size'),
               raw_names=('manufacturer_raw', lambda x: ' | '.join(sorted(set(map(str, x))))))
          .sort_values(['device_group', 'reports'], ascending=[True, False]))
pd.set_option('display.max_colwidth', 120)
print(review.groupby(level=0).head(8).to_string())

Distinct names: 142 raw -> 118 after cleaning

                                                                                  reports                                                                                                                                                                                                    raw_names
device_group                           manufacturer                                                                                                                                                                                                                                                   
Powered exercise equipment (incl. CPM) KLC 4038 WEAVER COURT E HILLIARD OH 43026       10                                                                                                                                                                KLC, 4038 WEAVER COURT E., HILLIARD, OH 43026
                                       MEDICA MEDIZINTECHNIK        

In [10]:
MANUAL = {
    'SHENZHEN DONGDIXIN TECHNOTOLOGY': 'SHENZHEN DONGDIXIN TECHNOLOGY',
    'REWALK ROBOTICS MARLBOROUGH MA': 'REWALK ROBOTICS',
    'LIFEWARD': 'REWALK ROBOTICS',
    'PARKER HUMAN MOTION & CONTROL': 'PARKER HANNIFIN',
    'PARKER HANNIFIN HUMAN MOTION AND CONTROL': 'PARKER HANNIFIN',
    'KLC 4038 WEAVER COURT E HILLIARD OH 43026': 'KLC',
    'BAYER HEALTHCARE CONSUMER CARE': 'BAYER HEALTHCARE',
    'OMRON HEALTHCARE': 'OMRON',
    'OMRON DALIAN PLANT 2': 'OMRON',
    'THE FURNISS': 'FURNISS',
    'MOTEK MEDICAL B V': 'MOTEK MEDICAL',
    'B TEMIA': 'B-TEMIA',
    'UNK': 'Not reported',
    'UNKNOWN': 'Not reported',
    'TENS': 'Not reported',
}
reports['manufacturer'] = reports['manufacturer'].replace(MANUAL)
print('Final distinct manufacturers:', reports['manufacturer'].nunique(), '\n')

top = (reports.groupby(['device_group', 'manufacturer']).size().rename('reports')
       .reset_index().sort_values(['device_group', 'reports'], ascending=[True, False]))
top['share_%'] = (100 * top['reports'] / top.groupby('device_group')['reports'].transform('sum')).round(1)
print(top.groupby('device_group').head(5).to_string(index=False))

Final distinct manufacturers: 108 

                          device_group                  manufacturer  reports  share_%
Powered exercise equipment (incl. CPM)                           KLC       11     27.5
Powered exercise equipment (incl. CPM)         MEDICA MEDIZINTECHNIK        5     12.5
Powered exercise equipment (incl. CPM)                        HOCOMA        4     10.0
Powered exercise equipment (incl. CPM)                       FURNISS        3      7.5
Powered exercise equipment (incl. CPM)                           DJO        2      5.0
                   Powered exoskeleton               REWALK ROBOTICS       37     68.5
                   Powered exoskeleton                  EKSO BIONICS        7     13.0
                   Powered exoskeleton                       B-TEMIA        3      5.6
                   Powered exoskeleton               PARKER HANNIFIN        3      5.6
                   Powered exoskeleton                   WANDERCRAFT        3      5.6
       

In [11]:
import os, sqlite3, shutil
C = f'{BASE}/data/clean'
os.makedirs(C, exist_ok=True)
group_of = reports['device_group']

dev_prob_clean = (dev_problems
    .merge(dev_lookup[['FDA_CODE', 'TERM']], left_on='DEVICE_PROBLEM_CODE', right_on='FDA_CODE', how='left')
    .rename(columns={'TERM': 'device_problem'}).drop(columns='FDA_CODE'))
dev_prob_clean['device_group'] = dev_prob_clean['MDR_REPORT_KEY'].map(group_of)

pat_prob_clean = (pat_problems
    .merge(pat_lookup[['FDA_CODE', 'TERM']], left_on='PROBLEM_CODE', right_on='FDA_CODE', how='left')
    .rename(columns={'TERM': 'patient_problem'}).drop(columns='FDA_CODE'))
pat_prob_clean['device_group'] = pat_prob_clean['MDR_REPORT_KEY'].map(group_of)

narr_clean = narratives.copy()
narr_clean['device_group'] = narr_clean['MDR_REPORT_KEY'].map(group_of)

print('Device problem codes with no description:', dev_prob_clean['device_problem'].isna().sum())
print('Patient problem codes with no description:', pat_prob_clean['patient_problem'].isna().sum(), '\n')

tables = {'reports': reports.reset_index(), 'device_problems': dev_prob_clean,
          'patient_problems': pat_prob_clean, 'narratives': narr_clean}

local_db = '/content/maude_rehab.db'
con = sqlite3.connect(local_db)
for name, df in tables.items():
    df.to_parquet(f'{C}/{name}.parquet', index=False)
    df.to_csv(f'{C}/{name}.csv', index=False)
    df.to_sql(name, con, if_exists='replace', index=False)
    print(f'saved {name}: {len(df):,} rows')

print('\nSQL check:')
print(pd.read_sql("""
    SELECT device_group,
           COUNT(*)             AS reports,
           SUM(serious_outcome) AS serious,
           SUM(severe_outcome)  AS severe
    FROM reports
    GROUP BY device_group
    ORDER BY reports DESC""", con))
con.close()
shutil.copy(local_db, f'{C}/maude_rehab.db')

Device problem codes with no description: 10
Patient problem codes with no description: 0 

saved reports: 477 rows
saved device_problems: 548 rows
saved patient_problems: 721 rows
saved narratives: 793 rows

SQL check:
                             device_group  reports  serious  severe
0                     TENS (prescription)      213       54      10
1                 TENS (over-the-counter)       97       30      18
2               Powered muscle stimulator       73       19      11
3                     Powered exoskeleton       54       42       7
4  Powered exercise equipment (incl. CPM)       40       15      11


'/content/drive/MyDrive/cs1-device-safety/data/clean/maude_rehab.db'